# 03. Experiment Tracking and Modeling

This notebook follows the Week 2 practical class flow, adapted to the Green Taxi project.

In the class notebooks, the workflow was:

1. Load a prepared dataset.
2. Import MLflow and set the tracking path.
3. Start an experiment.
4. Train a few models.
5. Log parameters, metrics, artifacts, and models.
6. Compare runs.
7. Use Optuna for hyperparameter tuning.

Here we use the same idea, but our target is `is_tipped`: whether a Green Taxi credit-card trip received a tip.

## Business Problem

The project wants to build a reproducible MLOps workflow using NYC Green Taxi data.

For this modeling notebook, the prediction task is:

> Predict whether a credit-card Green Taxi trip receives a tip.

This is a binary classification problem:

- `is_tipped = 1`: the passenger tipped.
- `is_tipped = 0`: the passenger did not tip.

## Loading the Data

Notebook 2 saves the prepared reference dataset to `data/02_intermediate/ref_data.parquet`.

This notebook starts from that prepared dataset. It does not repeat the raw data cleaning and feature engineering steps.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
from mlflow.models import infer_signature
from optuna.integration.mlflow import MLflowCallback

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_COL = "is_tipped"
MAX_ROWS_FOR_NOTEBOOK = 100_000

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_PATH = PROJECT_ROOT / "data" / "02_intermediate" / "ref_data.parquet"
REPORTING_DIR = PROJECT_ROOT / "data" / "08_reporting" / "modeling"
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing file: {DATA_PATH}. Run Notebook 2 first so it creates ref_data.parquet."
    )

df = pd.read_parquet(DATA_PATH)

if len(df) > MAX_ROWS_FOR_NOTEBOOK:
    df, _ = train_test_split(
        df,
        train_size=MAX_ROWS_FOR_NOTEBOOK,
        stratify=df[TARGET_COL],
        random_state=RANDOM_STATE,
    )
    df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)
display(df.head())
display(df[TARGET_COL].value_counts(normalize=True).rename("target_share").to_frame())

# Import mlflow and set the tracking to the right path

This mirrors the Week 2 notebook section where MLflow tracking is configured before starting experiments.

Instead of using a remote tracking server, we use a local `mlruns/` folder inside the project.

In [ ]:
TRACKING_DIR = PROJECT_ROOT / "mlruns"
mlflow.set_tracking_uri(TRACKING_DIR.as_uri())

EXPERIMENT_NAME = "green_taxi_is_tipped_week2_flow"
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment name:", EXPERIMENT_NAME)

# Starting an experiment

Before training models, we define `X` and `y`, split the data, and create one preprocessing object that all models will use.

This is the Green Taxi equivalent of preparing `X_train`, `X_test`, `y_train`, and `y_test` in the Week 2 notebooks.

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target mean in train:", round(y_train.mean(), 3))
print("Target mean in test:", round(y_test.mean(), 3))
print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

# Choose 3 models

The Week 2 problem notebook asks us to choose three models and log different metrics.

Here we use:

1. `DummyClassifier`: a trivial baseline.
2. `LogisticRegression`: a simple linear classification model.
3. `RandomForestClassifier`: a tree-based model.

The dummy model is important because it tells us whether the real models are actually better than a naive strategy.

In [ ]:
models = {
    "dummy_most_frequent": {
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"strategy": "most_frequent"},
        "description": "Naive baseline that always predicts the most frequent class.",
    },
    "logistic_regression": {
        "model": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        "params": {"max_iter": 1000, "class_weight": "balanced"},
        "description": "Simple linear classifier with balanced class weights.",
    },
    "random_forest": {
        "model": RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": 100,
            "max_depth": 10,
            "min_samples_leaf": 20,
            "class_weight": "balanced",
        },
        "description": "Tree-based model that can capture non-linear patterns.",
    },
}

# Logging relevant parameters with MLflow tracking

This section is the direct equivalent of the Week 2 MLflow logging cells.

For each model, we log:

- model parameters;
- classification metrics;
- a classification report artifact;
- a confusion matrix artifact;
- the trained sklearn pipeline.

In [ ]:
run_rows = []
trained_models = {}

for model_name, model_info in models.items():
    classifier = model_info["model"]
    params = model_info["params"]
    description = model_info["description"]

    full_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

    with mlflow.start_run(run_name=model_name, description=description) as run:
        full_model.fit(X_train, y_train)

        y_pred = full_model.predict(X_test)
        y_proba = full_model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(
            {
                "project": "green_taxi_mlops",
                "target": TARGET_COL,
                "task": "binary_classification",
                "stage": "baseline_model_comparison",
            }
        )

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        report_path = REPORTING_DIR / f"{model_name}_classification_report.json"
        with report_path.open("w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)
        mlflow.log_artifact(str(report_path), artifact_path="reports")

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False)
        ax.set_title(model_name)
        cm_path = REPORTING_DIR / f"{model_name}_confusion_matrix.png"
        fig.tight_layout()
        fig.savefig(cm_path, dpi=150)
        plt.close(fig)
        mlflow.log_artifact(str(cm_path), artifact_path="figures")

        signature = infer_signature(X_train.head(20), full_model.predict(X_train.head(20)))
        mlflow.sklearn.log_model(
            sk_model=full_model,
            artifact_path="model",
            signature=signature,
            input_example=X_train.head(3),
        )

        run_rows.append(
            {
                "model": model_name,
                "run_id": run.info.run_id,
                "artifact_uri": run.info.artifact_uri,
                "description": description,
                **metrics,
            }
        )
        trained_models[model_name] = full_model

runs_df = pd.DataFrame(run_rows).sort_values("f1", ascending=False)
display(runs_df)

# Create a pandas dataframe with experiment information

This mirrors the Week 2 exercise asking for a dataframe with experiment/run information.

The important columns are:

- `run_id`: MLflow run identifier;
- `artifact_uri`: where the model/artifacts were stored;
- metrics such as `f1` and `roc_auc`.

In [ ]:
baseline_comparison_path = REPORTING_DIR / "baseline_mlflow_runs.csv"
runs_df.to_csv(baseline_comparison_path, index=False)

display(runs_df[["model", "run_id", "artifact_uri", "accuracy", "precision", "recall", "f1", "roc_auc", "description"]])
print("Saved run comparison to:", baseline_comparison_path)

In [ ]:
metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]

ax = runs_df.set_index("model")[metric_columns].plot(
    kind="bar",
    figsize=(10, 4),
    ylim=(0, 1),
)
ax.set_title("Baseline model comparison")
ax.set_ylabel("Metric value")
ax.legend(loc="lower right")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# Hyperparameter fine-tuning with MLflow tracking

This mirrors the Optuna part of the Week 2 class.

Optuna tries different hyperparameter combinations. Each trial trains a model and returns one score. Here the score is `f1`, because this is a binary classification task and we want a balance between precision and recall.

We tune only the random forest because it was the most flexible baseline model.

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "max_depth": trial.suggest_int("max_depth", 4, 16),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(**params)),
        ]
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return f1_score(y_test, y_pred, zero_division=0)

In [ ]:
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="f1",
    mlflow_kwargs={"nested": True},
)

study = optuna.create_study(
    study_name="green_taxi_random_forest_f1",
    direction="maximize",
)

with mlflow.start_run(run_name="optuna_random_forest_parent") as parent_run:
    mlflow.set_tags(
        {
            "project": "green_taxi_mlops",
            "target": TARGET_COL,
            "stage": "optuna_tuning",
            "model_family": "random_forest",
        }
    )

    study.optimize(objective, n_trials=10, callbacks=[mlflow_callback])

    mlflow.log_metric("best_f1", study.best_value)
    mlflow.log_params({f"best_{key}": value for key, value in study.best_params.items()})

print("Best Optuna F1:", round(study.best_value, 4))
print("Best parameters:")
study.best_params

# Register your best model

In the class notebook, this section registers the selected model.

For this student project, we first log the best tuned model clearly. Registering it in the MLflow Model Registry can be added after the team agrees that this is the candidate model for serving.

In [ ]:
best_params = {
    **study.best_params,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

best_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(**best_params)),
    ]
)

with mlflow.start_run(run_name="best_optuna_random_forest") as run:
    best_model.fit(X_train, y_train)

    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    best_metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }

    mlflow.log_params(best_params)
    mlflow.log_metrics(best_metrics)
    mlflow.set_tags(
        {
            "project": "green_taxi_mlops",
            "target": TARGET_COL,
            "stage": "best_tuned_model",
            "optimizer": "optuna",
        }
    )

    signature = infer_signature(X_train.head(20), best_model.predict(X_train.head(20)))
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3),
    )

    best_run_id = run.info.run_id
    best_artifact_uri = run.info.artifact_uri

print("Best tuned model run id:", best_run_id)
print("Best tuned model artifact URI:", best_artifact_uri)
print(best_metrics)

# Final MLflow runs table

This final table keeps the same spirit as the Week 2 exercise: collect the most relevant experiment information in a pandas dataframe.

In [ ]:
best_row = {
    "model": "best_optuna_random_forest",
    "run_id": best_run_id,
    "artifact_uri": best_artifact_uri,
    "description": "Random forest tuned with Optuna.",
    **best_metrics,
}

final_runs_df = pd.concat(
    [runs_df, pd.DataFrame([best_row])],
    ignore_index=True,
).sort_values("f1", ascending=False)

final_runs_path = REPORTING_DIR / "final_mlflow_runs_summary.csv"
final_runs_df.to_csv(final_runs_path, index=False)

display(final_runs_df)
print("Saved final MLflow runs summary to:", final_runs_path)

## Conclusions

- This notebook follows the Week 2 class structure: MLflow setup, experiment start, three models, logged metrics/artifacts, run comparison, Optuna tuning, and best model logging.
- The Green Taxi adaptation uses `is_tipped` as the binary target.
- The most important comparison metric here is `f1`, because it balances precision and recall.
- The stable parts of this notebook can later be moved into a Kedro `model_train` pipeline.
- Before serving, the team still needs to confirm whether all features are available at the intended prediction time.